In [4]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel,  AutoModelForSeq2SeqLM
import pandas as pd


In [ ]:
!pip install bert-score
from bert_score import score

In [1]:
1

1

In [5]:
from tqdm import tqdm
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

In [6]:
from sentence_transformers import SentenceTransformer, util
paraph_model = SentenceTransformer('sentence-transformers/paraphrase-mpnet-base-v2').to(device)
#модель, на выходе которой эмбеддинг предложения, с косинусным сходством близкому к 1 для предложений похожих по смыслу


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
hf_model = paraph_model._first_module().auto_model
pooling = paraph_model._last_module()
ph_tokenizer = paraph_model._first_module().tokenizer
ph_emb_matrix = hf_model.get_input_embeddings().weight.data
emb_l = nn.Embedding.from_pretrained(ph_emb_matrix)
#inp_emb = emb_l(tokens['input_ids'])

In [ ]:
inp_emb.shape

In [57]:
inp_emb = emb_l(input_ids)
hidden = hf_model(inputs_embeds=inp_emb, attention_mask=attention_mask).last_hidden_state
sentence_emb2 = pooling({"token_embeddings": hidden, "attention_mask": attention_mask})["sentence_embedding"]

In [ ]:
!pip install lightning==2.4.0

In [7]:
import lightning as L

In [15]:
path = "/kaggle/input/financial-news-headlines/"
data_cnbc = pd.read_csv(path + "cnbc_headlines.csv").dropna()
data_guar =  pd.read_csv(path + "guardian_headlines.csv").dropna()
data_reut = pd.read_csv(path+ "reuters_headlines.csv").dropna()
data = pd.concat([data_reut['Headlines'],data_cnbc['Headlines'],data_guar['Headlines']],  ignore_index=True)

In [16]:
from sklearn.model_selection import train_test_split
train_texts, test_texts = train_test_split(data, test_size=0.1, random_state=42)

In [17]:
model_name = "yiyanghkust/finbert-tone"
#tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)

#EMB_MATRIX = bert_model.embeddings.word_embeddings.weight

In [18]:
class FinancialNewsDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        return text

train_data = FinancialNewsDataset(train_texts)
test_data = FinancialNewsDataset(test_texts)
len(train_texts), len(test_texts)

(48033, 5337)

In [19]:
def collate_fn(
    tokenizer: AutoTokenizer, batch: list[str]
) -> tuple[Tensor, Tensor]:
    encoded_batch = tokenizer.batch_encode_plus(
        batch, padding="longest", return_tensors="pt", return_token_type_ids=False)
    return encoded_batch.to(device)

len(ph_tokenizer)

30527

In [20]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=lambda batch:collate_fn(ph_tokenizer,batch))
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=lambda batch:collate_fn(ph_tokenizer,batch))
batch = next(iter(train_loader))
input_ids, attention_mask = batch['input_ids'], batch['attention_mask']
input_ids.shape

torch.Size([64, 25])

In [21]:
VOCAB_SIZE = len(ph_tokenizer)

In [32]:
def generate(probs, embedding_matrix=ph_emb_matrix, tokenizer=ph_tokenizer, T=1, n_samples=3):
  data = []
  for i in range(n_samples):
    p = (-torch.log(-torch.log(torch.rand(probs.shape,device=device))))
    alpha = F.softmax((p+torch.log(probs))/T, dim=-1)
    gen = torch.matmul(alpha, embedding_matrix)
    cls_tok = embedding_matrix[tokenizer.cls_token_id].contiguous().expand(gen.size(0),1,gen.size(-1)).contiguous().to(device)
    gen = torch.cat([cls_tok, gen], dim=1)
    data.append(gen)


  res = torch.stack(data,0)
  return res

model = EncoderDecoderModel(bert_model, ph_tokenizer, vocab_size=len(ph_tokenizer)).to(device)
logits, sent_emb = model(input_ids, attention_mask)
probs = F.softmax(logits[:,:-1,:], dim=-1)

In [32]:
g = torch.randn(16)
g

tensor([ 1.8990e-01,  4.3428e-01,  1.2900e+00,  3.6706e-01, -2.3169e+00,
         1.7798e+00, -1.2951e-01,  1.1469e+00,  4.5390e-01, -6.2855e-01,
         1.3416e+00,  8.1390e-01,  1.0972e+00,  2.2303e-02,  2.4258e+00,
        -1.2303e-03])

In [35]:
g.sum()

tensor(8.2865)

In [36]:
F.gumbel_softmax(g, hard=True).sum()

tensor(1.)

In [46]:
#оптимизуруем cos_sim между sent_emb1, sent_emb2,
#полученные из encoder для tokens и сгенерированных emb из выхода
def get_ph_emb(tokens):
    hidden = hf_model(input_ids=tokens['input_ids'], attention_mask=tokens['attention_mask']).last_hidden_state
    sentence_emb = pooling({"token_embeddings": hidden, "attention_mask": tokens['attention_mask']})["sentence_embedding"]
    return sentence_emb

def paraphrase_loss(logits, tokens):
  sent_emb = get_ph_emb(tokens)
  mask = tokens['attention_mask']
  y = F.gumbel_softmax(logits, tau=1.0, hard=True)
  gen = torch.matmul(y, ph_emb_matrix)
  cls_tok = ph_emb_matrix[ph_tokenizer.pad_token_id].expand(gen.size(0),1,gen.size(-1)).contiguous().to(device)
  gen = torch.cat([cls_tok, gen], dim=1)

  
  hidden = hf_model(inputs_embeds=gen, attention_mask=tokens['attention_mask']).last_hidden_state
  gen_sent_emb = pooling({"token_embeddings": hidden, "attention_mask": tokens['attention_mask']})["sentence_embedding"]
  loss = 1-(F.normalize(gen_sent_emb, dim=-1)*F.normalize(sent_emb, dim=-1)).sum(-1).mean()
  return loss

#model = EncoderDecoderModel(bert_model, ph_tokenizer, vocab_size=len(ph_tokenizer)).to(device)
logits, sent_emb = model.model.to(device)(input_ids, attention_mask)
#paraphrase_loss(logits[:,:-1,:], batch)
# def paraphrase_loss(gen_samples, tokens):
#   sent_emb = get_ph_emb(tokens)
#   mask = tokens['attention_mask']
#   gen_sent_embeds = []
#   for sample in gen_samples:
#     hidden = hf_model(inputs_embeds=sample, attention_mask=mask).last_hidden_state
#     sentence_emb = pooling({"token_embeddings": hidden, "attention_mask": mask})["sentence_embedding"]
#     gen_sent_embeds.append(sentence_emb)
#   gen_sent_embeds  = torch.stack(gen_sent_embeds,0)
#   loss = torch.tensor(1,device=device)-(F.normalize(gen_sent_embeds, dim=-1)*F.normalize(sent_emb, dim=-1)).sum(-1).mean()
#   return loss


#logits, sent_emb = model.model.cuda()(input_ids, attention_mask)
paraphrase_loss(logits[:,:-1,:], batch)

tensor(0.5369, device='cuda:0', grad_fn=<RsubBackward1>)

In [31]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, output_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            #nn.Dropout(p=0.01)
        )

    def forward(self, x, mask=None):
        attn_scores = self.attn(x).squeeze(-1)  # (B, T)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)  # (B, T)
        return torch.sum(attn_weights.unsqueeze(-1) * self.proj(x), dim=1)  # (B, H)


at = AttentionPooling(768, 1024).to(device)
x= bert_model(batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state
at(x).shape

torch.Size([64, 1024])

In [27]:
class FactorizedOutput(nn.Module):
    def __init__(self, hidden_dim, vocab_size, factor_dim):
        super(FactorizedOutput, self).__init__()
        self.dropout = nn.Dropout(0.5)
        self.proj_down = nn.Linear(hidden_dim, factor_dim, bias=False)
        self.proj_up = nn.Linear(factor_dim, vocab_size, bias=False)

    def forward(self, x):
        # Down-projection
        x = self.proj_down(x)  # (B, T, factor_dim)
        # Up-projection to vocab size
        logits = self.proj_up(self.dropout(x))  # (B, T, vocab_size)
        return logits

In [28]:
import math


class EncoderDecoderModel(nn.Module):
    def __init__(self, bert_model, tokenizer, hidden_dim=768, num_layers=2, nhead=8, max_length=1000, vocab_size=VOCAB_SIZE, dropout=1e-4, sent_dim=768):
        super().__init__()
        self.tokenizer = tokenizer
        self.bert = bert_model  # ЭНКОДЕР (BERT)
        self.bert.requires_grad_(False)
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        self.vocab_size = vocab_size
        self.sent_dim = sent_dim


        self.attn_pool = AttentionPooling(hidden_dim, sent_dim)


        self.tgt_projector= nn.Sequential(
            nn.Linear(hidden_dim, sent_dim),
            nn.LayerNorm(sent_dim),
            nn.GELU(),
            #nn.Dropout(p=0.01)
        )

        decoder_layer = nn.TransformerEncoderLayer(d_model=sent_dim, nhead=nhead,dim_feedforward=sent_dim,dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)

        # Выходной слой для предсказания токенов
        self.fc_out = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(sent_dim, vocab_size)
        )
        #self.factorized_output = FactorizedOutput(sent_dim, vocab_size, sent_dim//2)

        # Генерация синусоидальных позиционных эмбеддингов
        self.register_buffer("positional_encoding", self.sinusoidal_positional_encoding(max_length, sent_dim))
       
    def sinusoidal_positional_encoding(self, seq_length, hidden_dim):
        position = torch.arange(seq_length).unsqueeze(1).float()  # (T, 1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-math.log(10000.0) / hidden_dim))

        pe = torch.zeros(seq_length, hidden_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0).to(device)  # (1, T, H)

    def get_sentence_emb(self, input_ids, attention_mask):
        encoded = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state # (B, T, H)
        sent_embedding = self.attn_pool(encoded, attention_mask)
        return sent_embedding


    def forward(self, input_ids, attention_mask):
        batch_size, seq_length = input_ids.shape  # (B, T)

  
        sent_emb = self.get_sentence_emb(input_ids, attention_mask)  # (B, S)
        tgt = self.bert.embeddings.word_embeddings(input_ids)



        memory = sent_emb.unsqueeze(1)#torch.stack(memory, dim=1)
        #memory = sent_emb.unsqueeze(1).expand(batch_size, 20, -1)

        #memory = memory + self.positional_encoding[:, :20, :]

    
        # Декодерные входы (позиционные эмбеддинги)
        tgt = self.tgt_projector(tgt)

        # Добавляем синусоидальные позиции
        tgt = tgt + self.positional_encoding[:, :seq_length, :]  # (B, T, S)

        pref = sent_emb.unsqueeze(dim=1)
        tgt = torch.concat([pref, tgt], dim=1)

        # ПРОГОН ЧЕРЕЗ ДЕКОДЕР

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_length+1).to(device)

        attention_mask = torch.concat([torch.ones(batch_size, 1).cuda(), attention_mask], dim=1)
        # decoder_output = self.transformer_decoder(tgt,
        #                                           memory=memory,
        #                                           tgt_mask=tgt_mask.bool(),
        #                                           tgt_key_padding_mask=(attention_mask==0)
        #                                           )  # (B, T, S)
        decoder_output = self.transformer_decoder(tgt,
                                mask=tgt_mask.bool(),
                                src_key_padding_mask=(attention_mask==0)
                                )[:, 1:,:]



        # ПРОГОН ЧЕРЕЗ ВЫХОДНОЙ ЛИНЕЙНЫЙ СЛОЙ
        token_logits = self.fc_out(decoder_output)  # (B, T, V)

        return token_logits, sent_emb  # (B, T, V)

    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, max_len: int = 100):
        self.eval()

        B, S = sent_emb.shape

        memory = sent_emb.unsqueeze(1)

        tgt_ids = torch.full((B, 1), self.tokenizer.cls_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_len):
            t = tgt_ids.size(1)

            pos_emb = self.positional_encoding[:, :t, :]

            tgt = self.bert.embeddings.word_embeddings(tgt_ids)

            tgt = self.tgt_projector(tgt) + pos_emb
            pref = sent_emb.unsqueeze(dim=1)
            tgt = torch.concat([pref, tgt], dim=1)

            tgt_mask = nn.Transformer.generate_square_subsequent_mask(t+1).to(device)

            decoder_output = self.transformer_decoder(tgt,mask=tgt_mask.bool())[:, 1:,:]
            logits = self.fc_out(decoder_output[:,-1,:])

            next_token = torch.argmax(logits, dim=-1, keepdim=True)

            tgt_ids = torch.cat([tgt_ids, next_token], dim=-1)


            finished = finished | (next_token.squeeze(1) == self.tokenizer.sep_token_id)
            if finished.all():
                break


        return self.tokenizer.batch_decode(tgt_ids, skip_special_tokens=True)

In [58]:
1

1

In [72]:
class LightningAutoencoder(L.LightningModule):
    def __init__(self, model,  tokenizer, learning_rate=1e-4):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.tokenizer = tokenizer
        self.criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    def get_sentence_emb(self, batch):
        return self.model.get_sentence_emb(batch['input_ids'], batch['attention_mask'])

    def forward(self, batch):
        return self.model(batch['input_ids'], batch['attention_mask'])

    def training_step(self, batch, batch_idx):
        input_ids, attention_mask = batch['input_ids'], batch['attention_mask']

        outputs, sent_emb = self.model(input_ids, attention_mask)

        outputs = outputs[:,:-1,:]

        targets = input_ids[:, 1:].contiguous()  # Сдвигаем цель на 1 вправо

        ce_loss = self.criterion(outputs.reshape(-1, self.model.vocab_size), targets.reshape(-1))

        sent_sim_loss =paraphrase_loss(outputs, batch)

        loss =  sent_sim_loss

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

        accuracy = torch.logical_and(targets == outputs.argmax(dim=-1), attention_mask[:, 1:]).sum() / attention_mask[:, 1:].sum()
        self.log("train_accuracy", accuracy.item(), on_step=False, on_epoch=True, prog_bar=True)
        
        if batch_idx % 50==0:
            sent_emb = self.get_sentence_emb(batch)
            output_text = self.generate_from_embedding(sent_emb, max_len=100)
            input_text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)
            # P, R, F1 = score(output_text, input_text, lang='en', model_type='microsoft/deberta-xlarge-mnli')
            # self.log('val_BERTScore F1:', F1.mean().item(),on_step=False, on_epoch=True, prog_bar=True)
    
            in_embeddings = paraph_model.encode(input_text, convert_to_tensor=True)
            out_embeddings = paraph_model.encode(output_text, convert_to_tensor=True)
            ph = (F.normalize(in_embeddings,dim=-1)*F.normalize(out_embeddings,dim=-1)).sum(-1)
            #ph = torch.diagonal(util.pytorch_cos_sim(in_embeddings, out_embeddings))
            self.log("train_ph_sim", ph.mean(), on_step=False, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids, attention_mask = batch['input_ids'], batch['attention_mask']

        outputs, sent_emb = self.model(input_ids, attention_mask)

        outputs = outputs[:,:-1,:]

        targets = input_ids[:, 1:].contiguous()


        ce_loss = self.criterion(outputs.reshape(-1, self.model.vocab_size), targets.reshape(-1))

        sent_sim_loss = paraphrase_loss(outputs, batch)

        loss = sent_sim_loss

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    
        accuracy = torch.logical_and(targets == outputs.argmax(dim=-1), attention_mask[:, 1:]).sum() / attention_mask.sum()
        self.log("val_accuracy", accuracy.item(), on_step=False, on_epoch=True, prog_bar=True)

        #self.log("top_3", top_k(batch, k=3).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_5", top_k(batch, k=5).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_10", top_k(batch, k=10).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_15", top_k(batch, k=15).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_25", top_k(batch, k=25).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_50", top_k(batch, k=50).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_100", top_k(batch, k=100).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_250", top_k(batch, k=250).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_500", top_k(batch, k=500).item(), on_step=False, on_epoch=True, prog_bar=True)
        #self.log("top_1000", top_k(batch, k=1000).item(), on_step=False, on_epoch=True, prog_bar=True)



        sent_emb = self.get_sentence_emb(batch)
        output_text = self.generate_from_embedding(sent_emb, max_len=100)
        input_text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        # P, R, F1 = score(output_text, input_text, lang='en', model_type='microsoft/deberta-xlarge-mnli')
        # self.log('val_BERTScore F1:', F1.mean().item(),on_step=False, on_epoch=True, prog_bar=True)

        in_embeddings = paraph_model.encode(input_text, convert_to_tensor=True)
        out_embeddings = paraph_model.encode(output_text, convert_to_tensor=True)
        ph = (F.normalize(in_embeddings,dim=-1)*F.normalize(out_embeddings,dim=-1)).sum(-1)
        self.log("val_ph_sim", ph.mean(), on_step=False, on_epoch=True, prog_bar=True)

        return {
            "loss": loss,
            "preds": outputs,
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=0.01)
        # давайте кроме оптимизатора создадим ещё расписание для шага оптимизации
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.1, min_lr=1e-5,
                patience=5, verbose=True
            ),
            "monitor": "val_accuracy"
        }


    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, max_len: int = 100):
        self.eval()
        return self.model.generate_from_embedding(sent_emb, max_len)


In [34]:
def top_k(batch, k=5):
    attention_mask = batch['attention_mask']
    input_ids = batch['input_ids']
    logits,_  = model(batch)  # Замените B, T, V на соответствующие значения
    logits = logits[:,:-1,:]
    targets = input_ids[:,1:]  # Целевые индексы токенов

    # Получаем топ-5 предсказаний для каждого токена
    topk_probs, topk_indices = torch.topk(logits, k, dim=-1)  # top5_indices имеет форму [B, T, 5]

    # Сравниваем целевые токены с топ-5 предсказаниями
    # Добавляем дополнительное измерение к targets для сравнения
    targets_expanded = targets.unsqueeze(-1)  # Форма: [B, T, 1]

    # Проверяем, совпадает ли целевой токен с одним из топ-5 предсказаний
    correct = (topk_indices == targets_expanded).any(dim=-1)  # Форма: [B, T], значения True/False

    # Вычисляем точность: среднее значение по всем токенам
    topk_accuracy = torch.logical_and(correct, attention_mask[:, 1:]).sum() / attention_mask.sum()

    #print(f"Top-5 Accuracy: {top5_accuracy:.4f}")
    return topk_accuracy


In [ ]:
#!rm -rf /content/tb_logs

In [73]:
#hidden_dim=768, num_layers=2, nhead=8, max_length=100, vocab_size=len(tokenizer),dropout=1e-4, sent_dim=1024
#encoder_decoder_model = EncoderDecoderModel(bert_model, tokenizer=ph_tokenizer, sent_dim=768).to(device)
model = LightningAutoencoder(encoder_decoder_model, tokenizer=ph_tokenizer).to(device)
model.load_state_dict(torch.load('/kaggle/input/model-paraph/my_model.pth', weights_only=True))

<All keys matched successfully>

In [64]:
from lightning.pytorch.callbacks.model_summary import summarize

print(summarize(model,2))

  | Name                      | Type                | Params | Mode
-------------------------------------------------------------------------
0 | model                     | EncoderDecoderModel | 141 M  | eval
1 | model.bert                | BertModel           | 109 M  | eval
2 | model.attn_pool           | AttentionPooling    | 592 K  | eval
3 | model.tgt_projector       | Sequential          | 592 K  | eval
4 | model.transformer_decoder | TransformerEncoder  | 7.1 M  | eval
5 | model.fc_out              | Sequential          | 23.5 M | eval
6 | criterion                 | CrossEntropyLoss    | 0      | eval
-------------------------------------------------------------------------
31.8 M    Trainable params
109 M     Non-trainable params
141 M     Total params
566.021   Total estimated model params size (MB)
0         Modules in train mode
265       Modules in eval mode


In [74]:
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_loss:.2f}",
        monitor="val_loss",
        mode="min",
        save_top_k=3,
        save_last=True,
    )
]
trainer = L.Trainer(
     max_epochs=15,
     accelerator="auto",
     logger=TensorBoardLogger(save_dir="tb_logs", version='768'),
     callbacks=callbacks
)

INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir tb_logs

In [42]:
trainer.validate(model, test_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.10/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_accuracy        │    0.47314417362213135    │
│         val_loss          │    3.1161584854125977     │
│        val_ph_sim         │    0.4683191180229187     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 3.1161584854125977,
  'val_accuracy': 0.47314417362213135,
  'val_ph_sim': 0.4683191180229187}]

In [49]:
trainer.validate(model, test_loader)

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_accuracy        │    0.47314417362213135    │
│       val_loss_0001       │    0.5873523354530334     │
│    val_loss_1_samples1    │    0.5443251132965088     │
│   val_loss_1_samples10    │    0.5443267226219177     │
│    val_loss_1_samples3    │    0.5444899797439575     │
└───────────────────────────┴───────────────────────────┘

[{'val_accuracy': 0.47314417362213135,
  'val_loss_1_samples3': 0.5444899797439575,
  'val_loss_1_samples1': 0.5443251132965088,
  'val_loss_1_samples10': 0.5443267226219177,
  'val_loss_0001': 0.5873523354530334}]

In [75]:
trainer.fit(model, train_loader, test_loader)
#val_loss=2.600, val_accuracy=0.536, val_ph_sim=0.540, train_loss=1.070, train_accuracy=0.771, train_ph_sim=0.699

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name      | Type                | Params | Mode 
----------------------------------------------------------
0 | model     | EncoderDecoderModel | 141 M  | eval 
1 | criterion | CrossEntropyLoss    | 0      | train
----------------------------------------------------------
31.8 M    Trainable params
109 M     Non-trainable params
141 M     Total params
566.021   Total estimated model params size (MB)
1         Modules in train mode
264       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
#dim=2560, val_loss=2.310, val_accuracy=0.550, train_loss=1.190, train_accuracy=0.692
#dim = 768 v_num=768, val_loss=2.710, val_accuracy=0.503, val_BERTScore F1:=0.528, train_loss=2.080, train_accuracy=0.553
#dim=768 long_texts val_loss=2.650, val_accuracy=0.492, val_BERTScore F1:=0.471, train_loss=2.550, train_accuracy=0.491

In [ ]:
import torch
import torch.nn.functional as F

@torch.inference_mode()
def batch_beam_search_from_embedding_topk(model, sent_emb: torch.Tensor, beam_width=5, max_len=50):
    model.eval()
    device = sent_emb.device
    B = sent_emb.size(0)
    memory = sent_emb.unsqueeze(1)  # [B, 1, S]

    cls_id = model.tokenizer.cls_token_id
    sep_id = model.tokenizer.sep_token_id

    all_decoded_sequences = []

    for b in range(B):
        mem = memory[b:b+1]  # [1, 1, S]

        beams = [(torch.tensor([[cls_id]], device=device), 0.0)]  # list of (seq, score)

        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[0, -1].item() == sep_id:
                    candidates.append((seq, score))
                    continue

                t = seq.size(1)
                pos_emb = model.positional_encoding[:, :t, :]  # [1, T, D]
                tgt = model.bert.embeddings.word_embeddings(seq)  # [1, T, D]
                tgt = model.tgt_projector(tgt) + pos_emb
                tgt_mask = torch.nn.Transformer.generate_square_subsequent_mask(t).to(device)

                decoder_output = model.transformer_decoder(tgt,
                                                           memory=mem,
                                                           tgt_mask=tgt_mask.bool())  # [1, T, D]
                logits = model.fc_out(decoder_output[:, -1, :])  # [1, vocab]
                log_probs = F.log_softmax(logits, dim=-1)  # [1, vocab]

                topk_log_probs, topk_ids = torch.topk(log_probs, beam_width, dim=-1)  # [1, K]

                for k in range(beam_width):
                    next_token_id = topk_ids[0, k].unsqueeze(0).unsqueeze(0)  # [1, 1]
                    new_seq = torch.cat([seq, next_token_id], dim=1)  # [1, T+1]
                    new_score = score + topk_log_probs[0, k].item()
                    candidates.append((new_seq, new_score))

            # отбираем top-k лучших
            beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]

            # если все beams уже закончились — можно выйти раньше
            if all(seq[0, -1].item() == sep_id for seq, _ in beams):
                break

        # декодируем все top-k гипотез
        decoded_k = [
            model.tokenizer.decode(seq[0], skip_special_tokens=True)
            for seq, _ in beams
        ]
        all_decoded_sequences.append(decoded_k)

    return all_decoded_sequences  # list of B items, each is list of K strings


In [ ]:
model.to(device)
batch_iter = iter(test_loader)

In [ ]:
def get_input_text(batch):
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    input_text = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
    return  input_text

In [ ]:
def res_text(model, batch, beam=True, stoch=False):
    input_text =get_input_text(batch)
    sent_emb = model.get_sentence_emb(batch)
    if stoch:
      sent_emb+=torch.randn_like(sent_emb)
    if beam:
      output_text = batch_beam_search_from_embedding_topk(model.model, sent_emb)
    else:
      output_text = model.model.generate_from_embedding(sent_emb, max_len=1000)
    return list(zip(input_text, output_text))

In [ ]:
res_text(model, batch, beam=False)

In [ ]:
res_text(model, batch, beam=False, stoch=True)

In [ ]:
def f(res):
  in_embeddings = paraph_model.encode([res[0]], convert_to_tensor=True)
  out_embeddings = paraph_model.encode(res[1], convert_to_tensor=True)
  ph = (F.normalize(in_embeddings,dim=-1)*F.normalize(out_embeddings,dim=-1)).sum(-1)
  return torch.max(util.pytorch_cos_sim(in_embeddings, out_embeddings))

In [ ]:
input_text =get_input_text(batch)
sent_emb = model.get_sentence_emb(batch)
sent_emb_stoch = sent_emb + torch.randn_like(sent_emb)

In [ ]:
f(res)

In [ ]:
def get_bert_score(model, batch, stoch=False):
  input_text =get_input_text(batch)
  sent_emb = model.get_sentence_emb(batch)
  if stoch:
    sent_emb+=torch.randn_like(sent_emb)
  output_text = model.model.generate_from_embedding(sent_emb, max_len=1000, top_k=1)
  P, R, F1 = score(output_text, input_text, lang='en', model_type='microsoft/deberta-xlarge-mnli')
  return F1.mean().item()

In [ ]:
get_bert_score(model, batch, stoch=True)

In [ ]:
get_bert_score(model, batch)

In [ ]:
trainer = L.Trainer(
    accelerator="auto",
)


In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, max_length=100, sent_dim=2560).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)
#checkpoint_path = next(Path(f"/content/tb_logs/lightning_logs/version_1/checkpoints/").glob("*.ckpt"))
model.load_state_dict(torch.load('tb_logs/lightning_logs/768/checkpoints/last.ckpt', weights_only=True)["state_dict"])
#model.load_state_dict(torch.load('/content/drive/MyDrive/models/Model2560.pth', weights_only=True))

In [ ]:
model.model.sent_dim

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, sent_dim=768,max_length=1000).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)
#checkpoint_path = next(Path(f"/content/tb_logs/lightning_logs/version_1/checkpoints/").glob("*.ckpt"))
model.load_state_dict(torch.load('tb_logs/lightning_logs/768/checkpoints/last-v1.ckpt', weights_only=True)["state_dict"])
#model.load_state_dict(torch.load('/content/drive/MyDrive/models/Model768.pth', weights_only=True))

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

In [ ]:
trainer.validate(
    model,
    test_loader
    #ckpt_path=last_checkpoint_path,
)

# CLS ONLY

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model, sent_dim=2560).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)

last_checkpoint_path = Path("/content/tb_logs/lightning_logs/version_1/checkpoints/last.ckpt")
trainer.validate(
    model,
    test_loader,
    ckpt_path=last_checkpoint_path,
)

# ENCODER ONLY

In [ ]:
import math


class EncoderDecoderModel2(nn.Module):
    def __init__(self, bert_model, hidden_dim=768, num_layers=2, nhead=8, max_length=100, vocab_size=len(tokenizer),dropout=1e-3, sent_dim=1568, tokenizer=tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.bert = bert_model  # ЭНКОДЕР (BERT)
        self.bert.requires_grad_(False)
        self.hidden_dim = hidden_dim
        self.max_length = max_length
        self.vocab_size = vocab_size
        self.sent_dim = sent_dim


        self.attn_pool = AttentionPooling(hidden_dim, sent_dim)

        self.act = nn.GELU()


        self.pos_lin1 = nn.Linear(hidden_dim, hidden_dim)
        #self.pos_lin2 = nn.Linear(hidden_dim, sent_dim)

        # Декодер
        encoder_layer = nn.TransformerEncoderLayer(d_model=sent_dim, nhead=nhead,dim_feedforward=sent_dim,dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Выходной слой для предсказания токенов
        #self.fc_out = nn.Linear(sent_dim, vocab_size)
        self.factorized_output = nn.Linear(sent_dim, vocab_size)#FactorizedOutput(sent_dim, vocab_size, sent_dim//2)

        # Генерация синусоидальных позиционных эмбеддингов
        self.pos_proj2 = nn.Sequential(
            nn.Linear(hidden_dim, sent_dim),
            nn.GELU(),
            nn.LayerNorm(sent_dim),
            nn.Dropout(p=0.001)
        )

        self.register_buffer("positional_encoding", self.sinusoidal_positional_encoding(max_length, hidden_dim))

    def sinusoidal_positional_encoding(self, seq_length, hidden_dim):
        position = torch.arange(seq_length).unsqueeze(1).float()  # (T, 1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2).float() * (-math.log(10000.0) / hidden_dim))

        pe = torch.zeros(seq_length, hidden_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        return pe.unsqueeze(0).to(device)  # (1, T, H)

    def get_sentence_emb(self, input_ids, attention_mask):
        encoded = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state # (B, T, H)
        pos = self.positional_encoding[:, :input_ids.size(1), :]
        encoded += self.act(self.pos_lin1(pos))
        sent_embedding = self.attn_pool(encoded, attention_mask)
        return sent_embedding


    def forward(self, input_ids, attention_mask):
        batch_size, seq_length = input_ids.shape  # (B, T)

        with torch.no_grad():
            sent_emb = self.get_sentence_emb(input_ids, attention_mask)  # (B, S)
            tgt = self.bert.embeddings.word_embeddings(input_ids)



        # Добавляем синусоидальные позиции
        src = sent_emb.unsqueeze(1).expand(batch_size, seq_length, self.sent_dim)

        pos = self.positional_encoding[:, :seq_length, :]
        src = src + self.pos_proj2(pos)  # (B, T, S)

        # ПРОГОН ЧЕРЕЗ ДЕКОДЕР



        encoder_output = self.transformer_encoder.forward(src, src_key_padding_mask=(attention_mask==0))



        # ПРОГОН ЧЕРЕЗ ВЫХОДНОЙ ЛИНЕЙНЫЙ СЛОЙ
        token_logits = self.factorized_output(encoder_output)  # (B, T, V)

        return token_logits  # (B, T, V)

    @torch.inference_mode()
    def generate_from_embedding(self, sent_emb: torch.Tensor, mask, max_len: int = 25):
        self.eval()

        B, S = sent_emb.shape

        src = sent_emb.unsqueeze(1).expand(B, max_len, S)

        pos = self.positional_encoding[:, :max_len, :]
        src = src + self.pos_proj2(pos)

        encoder_output = self.transformer_encoder.forward(src)

        token_logits = self.factorized_output(encoder_output)
        out_tokens = token_logits.argmax(dim=-1)

        return self.tokenizer.batch_decode(out_tokens , skip_special_tokens=True), token_logits

In [ ]:
! ls tb_logs/lightning_logs/768/checkpoints/

In [ ]:
encoder_decoder_model = EncoderDecoderModel(bert_model).to(device)
model = LightningAutoencoder(encoder_decoder_model).to(device)

last_checkpoint_path = Path("tb_logs/lightning_logs/768/checkpoints/last-v1.ckpt")
trainer.validate(
    model,
    test_loader,
    ckpt_path=last_checkpoint_path,
)